In [ ]:
#Python File
#Imports
import cv2
import mediapipe as mp
import math
import time
import csv
import os
import pandas as pd
import numpy as np
import joblib
from scipy.interpolate import CubicSpline
from scipy.stats import pearsonr

det_model = joblib.load('detection_model_V2.pkl')
det_scaler = joblib.load('detection_scaler_V2.pkl')
scaler = joblib.load('scale_model_V7_optimized.pkl')
model = joblib.load('blink_model_V7_optimized.pkl')

scaler.feature_names_in_ = None
model.feature_names_in_ = None
det_scaler.feature_names_in_ = None
det_model.feature_names_in_ = None

if os.path.exists('report.csv'):
    os.remove('report.csv')

from scipy.interpolate import CubicSpline

def get_blink_physics_spline(times, values, min_EAR_during_blink, baseline_EAR):
    """
    Analyzes the blink using Cubic Spline derivatives for high-precision 
    velocity, acceleration, and duration.
    """
    try:
        times_arr, indices = np.unique(times, return_index=True)
        values_arr = np.array(values)[indices]
        
        if len(times_arr) < 4: 
            return None
        
        # Create the spline
        spline = CubicSpline(times_arr, values_arr, bc_type='natural')
        
        # 1ms Resolution interpolation
        res = 1000 
        hi_res_times = np.linspace(times_arr[0], times_arr[-1], num=int((times_arr[-1]-times_arr[0])*res))
        hi_res_values = spline(hi_res_times)
        
        # Calculate Derivatives (Velocity and Acceleration)
        velocities = spline(hi_res_times, 1)
        accelerations = spline(hi_res_times, 2)

        #Calculating the time at peak
        peak_idx = np.argmin(hi_res_values)
        peak_ts_precise = hi_res_times[peak_idx]
        
        # Precise Duration Logic (10% threshold above minimum)
        strict_threshold = min_EAR_during_blink + 0.10 * (baseline_EAR - min_EAR_during_blink)
        below = hi_res_times[hi_res_values <= strict_threshold]
        
        duration = below[-1] - below[0] if len(below) > 1 else (times_arr[-1] - times_arr[0])
        
        return {
            "duration": duration,
            "peak_closing_velocity": np.min(velocities),
            "peak_ts_precise": peak_ts_precise,
            "max_acceleration": np.max(np.abs(accelerations)),
            "start_ts_precise": below[0] if len(below) > 0 else times_arr[0],
            "end_ts_precise": below[-1] if len(below) > 0 else times_arr[-1]
        }
    except:
        return None


#Intensity calculator using predefined and well known ITU-R 601 formula
def get_pixel_intensity(pixel_bgr):
    #unpack pixel_bgr
    b, g, r = pixel_bgr
    #the standard Luminance formula
    intensity = 0.299*r + 0.587*g + 0.114*b
    return intensity


def create_file(filename):
    
    with open(filename, mode = 'w', newline = '') as f:
        writer = csv.writer(f)
        writer.writerow(["Timestamp", "EAR", "EAR Delta", "Velocity", "Acceleration", "BlinkLabel"])


def write_to_file(filename, TimeStamp, EAR_Val = 0 , change = 0, velocity = 0, acceleration = 0,  Blink_Label = 0):
    with open(filename, mode = 'a', newline = '') as f:
        writer = csv.writer(f)
        writer.writerow([TimeStamp, EAR_Val, change,velocity, acceleration,  Blink_Label])

#Timer Starter
start_time = time.time()

#EAR Calculator 
def EARratio(coords):
    w = math.dist(coords[0],coords[1])
    h1 = math.dist(coords[2],coords[3])
    h2 = math.dist(coords[4],coords[5])
    EAR = (h1 + h2)/(w*2)
    return EAR

#Two Key functions
mp_face_mesh = mp.solutions.face_mesh

#Initialising Key Values
blinked_frames = 0
blink_count = 0
blink_start = 0
#blink_end = 0
intersaltatory_intervals = []
blink_durations = []
all_min_EARs = []
instantaneous_rates = []
cumulative_rates = []
right_indexes = [362, 263, 385, 380, 387, 373]  
left_indexes = [33, 133, 160, 144, 158, 153] 
min_EAR_in_blink = 1.0
prev_EAR = 0
prev_velocity = 0
#closing_start_time = 0
time_of_min_ear = 0
#is_closing_phase = False
closing_durations = []

# Latest solution idea (Using buffer)
# (Keep your existing variables, but add these)
ear_buffer = []
time_buffer = []
buffer_size = 25 # Store ~0.83 s of data to capture full blink curve
baseline_ear = 0.3
threshold_start = 0.25

# Add new lists to store the spline-calculated physics
spline_peak_velocities = []
spline_peak_accelerations = []

 
#Begins recording
cap = cv2.VideoCapture(0)
'''cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
'''
with mp_face_mesh.FaceMesh(max_num_faces = 1, refine_landmarks = True, min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as face_mesh:
    
    #The Light requirements - to be modified
    light_crit_low = 40
    light_low = 55
    light_high = 160
    light_crit_high = 210
    valid_dist_center = 0.20
    stable_time = -1
    stable_start = 2
    light_eligible = False
    stable = False

    while cap.isOpened():
        rec, frame = cap.read()
        
        if not rec:
            continue
        
        flipped_frame = cv2.flip(frame, 1)
        h, w, _ = flipped_frame.shape

        # Convert for MediaPipe
        rgb_frame = cv2.cvtColor(flipped_frame, cv2.COLOR_BGR2RGB)
        flipped_frame.flags.writeable = False
        results = face_mesh.process(rgb_frame)
        flipped_frame.flags.writeable = True
        flipped_frame = cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR)

        brightness = 0
        face_centered = False

        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0]

            #Face centering system
            nose = landmarks.landmark[0]
            face_x, face_y = nose.x, nose.y

            dist_from_center = math.dist((face_x,face_y),(0.5,0.5))
            
            #Eye lighting check system
            eye_intensities = []
            drawing_points = []
            for point_counter in right_indexes + left_indexes:
                #Extracting each index of the eyes
                point = landmarks.landmark[point_counter]
                #converting mediapipe float to pixel value
                cx_raw, cy_raw = int(point.x*w), int(point.y*h)
                #Ensuring pixel value is within bounds
                cx, cy = max(0, min(cx_raw, w - 1)), max(0, min(cy_raw, h - 1))
                #Draws a circle at the eye points
                cv2.circle(flipped_frame, (cx,cy), 2, (40,255,0), 1)
                drawing_points.append((point.x, point.y))
                #Extracts bgr list for the colours at the point using Matrix Notation
                colour_at_point = flipped_frame[cy, cx]

                eye_intensities.append(get_pixel_intensity(colour_at_point))

            eye_brightness = np.mean(eye_intensities)

            #Brightness bar
            bar_height = int((eye_brightness / 255) * 200)
            cv2.rectangle(flipped_frame, (w-50, h-50), (w-30, h-250), (50, 50, 50), -1)
            cv2.rectangle(flipped_frame, (w-50, h-50), (w-30, h-50-bar_height), (255, 255, 0), -1) 
            cv2.putText(flipped_frame, "BRIGHT", (w-80, h-260), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

            if dist_from_center <= valid_dist_center:
                #Draw a rectangle showing the 40-60% zone
                start_point = (int(w * 0.3), int(h * 0.2))
                end_point = (int(w * 0.7), int(h * 0.8))
                cv2.rectangle(flipped_frame, start_point, end_point, (0, 255, 0), 1)
          
            #Determine the status
            if dist_from_center > valid_dist_center:
                #Draw a rectangle showing the 40-60% zone
                start_point = (int(w * 0.3), int(h * 0.2))
                end_point = (int(w * 0.7), int(h * 0.8))
                cv2.rectangle(flipped_frame, start_point, end_point, (0, 0, 255), 1)
                alert = "Face not centered, move face toward middle of the frame"
                color = (0, 0, 255) 
                light_eligible = False
            #Red Condition
            elif eye_brightness <= light_crit_low:
                alert = "CRITICAL: Too dark to detect eyes"
                color = (0, 0, 255) 
                light_eligible = False
            #Yellow condition
            elif light_crit_low < eye_brightness <= light_low:
                alert = "Slightly Low Light: Brighten screen or environment"
                color = (0, 255, 255)
                light_eligible = False
            #Green Condition
            elif light_low < eye_brightness < light_high:
                alert = "PERFECT CONDITIONS: Ready to test"
                color = (0, 255, 0)
                light_eligible = True

            #Orange Condition
            elif light_high < eye_brightness <= light_crit_high:
                alert = "Slightly High Light: May cause error in reflex test"
                color = (0, 165, 255)
                light_eligible = False

            #Red condition
            elif eye_brightness > light_crit_high: 
                alert = "CRITICAL: Light too bright (Washed out)"
                color = (0, 0, 255)
                light_eligible = False
            else:
                pass
            
            #Timer system for light and centralising stability
            if light_eligible:
                if stable_time == -1:
                    stable_time = time.time()
                elapsed = time.time() - stable_time

                if elapsed > stable_start:
                    alert = "STABLE! Press 'c' to start"
                    color = (0,255,0)
                    stable = True
                else:
                    alert = f"HOLD STILL for {round(stable_start - elapsed, 1)}s more"
                    stable = False
            else:
                stable_time = -1
                stable = False

            cv2.putText(flipped_frame, alert, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
            #Display lightning check
            
        else:
            cv2.putText(flipped_frame, "FACE NOT FOUND", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            stable_time = -1

        cv2.imshow("Lighting Optimisation", flipped_frame)
        #cv2 safety net
        key = cv2.waitKey(1) & 0xff

        if stable and key == ord('c'):
            break
        elif key == ord('x'):
            cap.release()
            cv2.destroyAllWindows()
            exit()


cv2.destroyWindow("Lighting Optimisation")

time.sleep(2)

#Stopwatch reset
start_time = time.time()
last_blink_end = time.time() 
prev_frame_time = time.time()

#Using FaceMesh class
with mp_face_mesh.FaceMesh(max_num_faces = 1, refine_landmarks = True, min_detection_confidence = 0.5, min_tracking_confidence = 0.5) as face_mesh:

    create_file('blinks.csv')
    #Video Loop
    while cap.isOpened():

        #Read video data
        rec, frame = cap.read()
        
        #Avoiding webcam recording inconsistency
        if not rec:
            continue

        #Flipping to ensure mirror-like display
        flipped_vid = cv2.flip(frame, 1)

        #Converting BGR to RGB because mediapipe uses RGB but cv2 uses BGR
        video = cv2.cvtColor(flipped_vid, cv2.COLOR_BGR2RGB)
        
        #Making the colour swapped image unwriteable to make program more efficient and faster to read
        video.flags.writeable = False
        result = face_mesh.process(video)
        
        #Converting colour code format back to cv2 format (BGR)
        video.flags.writeable = True
        video = cv2.cvtColor(video, cv2.COLOR_RGB2BGR)

        #Calculates dimensions
        height, width, _ = video.shape
        
        #Condition just to avoid tracking errors
        if result.multi_face_landmarks:

            face_landmark = result.multi_face_landmarks[0]
            #Loop to draw a mesh and for values
        
            #Getting the x and y's of the indexes
            R_XYs = [(face_landmark.landmark[r].x, face_landmark.landmark[r].y) for r in right_indexes]
            L_XYs = [(face_landmark.landmark[l].x, face_landmark.landmark[l].y) for l in left_indexes]
            
            #EAR Calculations
            avg_EAR = (EARratio(R_XYs) + EARratio(L_XYs)) / 2

            # INITIALIZE PHYSICS VARIABLES
            current_velocity = 0
            crnt_acceleration = 0

            current_time_wall = time.time() # This is our "Master Clock"
            time_since_start = current_time_wall - start_time

            ear_buffer.append(avg_EAR)
            time_buffer.append(current_time_wall) # Use the raw wall time here
            
            #Ear buffer list new Method ##############################################
            if len(ear_buffer) > buffer_size:
                ear_buffer.pop(0)
                time_buffer.pop(0)

            #Velocity and Acceleration current splicing new method
            if len(ear_buffer) >= 3:
                dt = time_buffer[-1] - time_buffer[-2]
                if dt > 0:
                    # Velocity: (Current EAR - Previous EAR) / Time
                    current_velocity = (ear_buffer[-1] - ear_buffer[-2]) / dt
                    
                    # Acceleration: (Current Vel - Previous Vel) / Time
                    # We need at least 3 points for this
                    prev_dt = time_buffer[-2] - time_buffer[-3]
                    prev_velocity = (ear_buffer[-2] - ear_buffer[-3]) / prev_dt
                    crnt_acceleration = (current_velocity - prev_velocity) / dt
                else:
                    current_velocity = 0
                    crnt_acceleration = 0


            if prev_EAR == 0:
                prev_EAR = avg_EAR
                continue # Skip one frame to establish a baseline

            #Time difference (dt)
            current_frame_time = time.time()
            
            #Displacement (EAR Delta)
            del_EAR = avg_EAR - prev_EAR
            prev_EAR = avg_EAR # Update for next frame

            #AI Predictor
            clues = np.array([[avg_EAR, current_velocity, crnt_acceleration]])
            scld_clues = scaler.transform(clues)
            prediction = model.predict(scld_clues)[0]

            #Get Probability instead of just 0 or 1
            #proba = model.predict(scld_clues)[0][1] 

            #Use a custom threshold (you can lower this to 0.3 if it's not sensitive enough)
            #prediction = 1 if proba > 0.5 else 0

            
            # Update baseline only when eyes are open (prediction == 0)
            # This solves your "distance from camera" concern
            if prediction == 0 and len(ear_buffer) > 5:
                baseline_ear = np.mean(ear_buffer[-5:])
                #threshold_start = baseline_ear * 0.85 # 85% of their normal open eye 

            #AI strategy of counting a blink when eyes go down(1)
            if prediction == 1:
                if blinked_frames == 0:
                    #Count
                    blink_count += 1

                    blink_start = current_time_wall # Used for gap duration
                    time_of_min_ear = current_time_wall
                    actual_blink_start = time_since_start # Used for duration

                    #Latest min EAR 
                    min_EAR_in_blink = avg_EAR

                    #Rudimentary UI
                    #cv2.putText(video, "BLINK!", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
                
                #Frame incrementation
                blinked_frames += 1
                blink_dur = -1

                #track the lowest EAR seen during this specific blink
                if avg_EAR < min_EAR_in_blink:
                    min_EAR_in_blink = avg_EAR
                    time_of_min_ear = current_time_wall
            else:
                if blinked_frames > 0:
                    
                    #Only send the frames belonging to the current blink to the Spline
                    #Grab blink window from buffer
                    num_frames = min(blinked_frames + 4, len(ear_buffer)) # +4 to catch the 'tail' of the blink
                    blink_times = list(time_buffer)[-num_frames:]
                    blink_EARs = list(ear_buffer)[-num_frames:]

                    # Append the PRECISE duration to your list
                    physics = get_blink_physics_spline(blink_times, blink_EARs, min_EAR_in_blink, baseline_ear)

                    if physics:
                        gap_duration = physics["start_ts_precise"] - last_blink_end
                        
                        # Check if this is the FIRST blink to avoid NameError
                        is_first_blink = len(blink_durations) == 0

                        if gap_duration < 0.15 and not is_first_blink:
                            # MERGE LOGIC
                            new_total_dur = (physics["end_ts_precise"] - last_blink_start_timestamp)
                            blink_durations[-1] = new_total_dur
                            all_min_EARs[-1] = min(all_min_EARs[-1], min_EAR_in_blink)

                            # Update Cumulative Rate for the merged blink using precise end time
                            total_mins = (physics["end_ts_precise"] - start_time) / 60.0
                            if total_mins > 0:
                                cumulative_rates[-1] = len(blink_durations) / total_mins
                            
                        else:
                            
                            blink_durations.append(physics["duration"])
                            all_min_EARs.append(min_EAR_in_blink) 

                            #Closing duration calculation
                            closing_durations.append(physics["peak_ts_precise"] - physics["start_ts_precise"])

                            #Calculating inter-blink intervals
                            intersaltatory_intervals.append(max(gap_duration, 0))

                            #Appending peak velocities and accelerations
                            spline_peak_velocities.append(abs(physics["peak_closing_velocity"]))
                            spline_peak_accelerations.append(physics["max_acceleration"])
                            
                            last_blink_start_timestamp = physics["start_ts_precise"]
                            
                            inst_rate = 60 / gap_duration if gap_duration > 0 else 0
                            instantaneous_rates.append(inst_rate)

                            # Precise Cumulative Rate (Total Blinks / Total Minutes elapsed)
                            total_mins = (physics["end_ts_precise"] - start_time) / 60.0
                            cumulative_rates.append(len(blink_durations) / total_mins if total_mins > 0 else 0)

                        last_blink_end = physics["end_ts_precise"]

                    # Reset
                    min_EAR_in_blink = 1.0
                    
                    #Reset counter
                    blinked_frames = 0
                else:
                    #Old logic
                    #blink_dur = -1
                    min_EAR_in_blink = 1.0
            
            #Fills CSV File
            write_to_file('blinks.csv', time_since_start, avg_EAR, del_EAR, current_velocity, crnt_acceleration, prediction)

            #Blink count on screen for debugging
            cv2.putText(video, str(blink_count), (560, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
        else:
             cv2.putText(flipped_frame, "FACE NOT FOUND", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)      
        
        #Temporary timer
        if time.time() - start_time > 150:
            break

        #Closes code when x is pressed
        if cv2.waitKey(1) & 0xFF == ord('x'):
            break

        #Displays video with mesh
        cv2.imshow("Blink Detection", video)

#Calculates end time
end_time = time.time()

#Neccesary to stop using webcam
cap.release()
    
#Closes webcam window
cv2.destroyAllWindows()

#Total time calculator
total_time = end_time - start_time

#Blink Summary
if blink_count > 0:
    # Ensure all lists match the same length (protects against list mismatch errors)
    min_length = min(len(blink_durations), 
                     len(intersaltatory_intervals), 
                     len(spline_peak_velocities), 
                     len(cumulative_rates))
    
    blink_rate = blink_count/total_time

    report_dict = {
        'Blink ID': range(1, min_length + 1),
        'Duration (ms)': [d*1000 for d in blink_durations[:min_length]],
        'Closing Phase (ms)': [d*1000 for d in closing_durations[:min_length]],
        'Time since last (ms)': [d*1000 for d in intersaltatory_intervals[:min_length]],
        'Peak Velocity (units/s)': spline_peak_velocities[:min_length],
        'Peak Acceleration (units/s^2)': spline_peak_accelerations[:min_length],
        'Instantaneous Blink Rate (bpm)': instantaneous_rates[:min_length],
        'Cumulative Rates Blink Rate (bpm)': cumulative_rates[:min_length],
        'Min EAR Closure': all_min_EARs[:min_length]
    }

    df_report = pd.DataFrame(report_dict)
    df_report.to_csv('report.csv', index=False)
    
    # Calculate Averages using Spline Values
    avg_peak_velocity = round(np.mean(spline_peak_velocities), 3)
    avg_blink = round(np.mean(blink_durations), 5)
    
    print(f"Average Blink Rate: {(blink_count / (end_time - start_time)) * 60:.2f} blinks/min")
    print(f"Precise Average Duration: {avg_blink*1000:.2f} ms")
    print(f"Refined Peak Velocity: {avg_peak_velocity} units/sec")


#Enough blinks to be statistically "Safe"
if len(blink_durations) >= 5:
    #Use the last 10 blinks (or all if less than 10) for the most current state
    window = min(len(blink_durations),10)
    recent_durations = [d * 1000 for d in blink_durations[-window:]] 
    recent_rates = instantaneous_rates[-window:]
    
    #Calculate the inputs for ML Model
    avg_dur = np.mean(recent_durations)
    avg_rate = np.mean(recent_rates)
    
    # Run the Predictor
    det_clues = pd.DataFrame([[avg_rate, avg_dur]], columns=['Blink Rate (bpm)', 'Blink Duration (ms)'])
    scld_det_clues = det_scaler.transform(det_clues)
    possibility = det_model.predict_proba(scld_det_clues)
    

    print(f"Final Probability: {round(possibility[0][1]*100, 2)}%")

    #Statistical Significance (The Correlation Check)
    if len(blink_durations) >= 8:
        
        # We correlate the last 15 blinks to see the immediate trend
        corr, p_val = pearsonr(instantaneous_rates[-15:], blink_durations[-15:])
        
        print(f"Correlation (Rate vs Duration): {round(corr, 3)}")
        if p_val < 0.05:
            print(" Statistical Significance (Valid Trend)")
        else:
            print("Trend is not yet statistically significant.")
else:
    print("Session too short")




'''# --- REFINED HEURISTIC CHECKING ---

# 1. Load your data
df = pd.read_csv('blinks.csv')

# 2. Define your "Physics Thresholds" 
# These represent the 'minimum effort' a real blink must exert
MIN_VELOCITY_FOR_BLINK = 4.0
MIN_ACCEL_FOR_BLINK = 75.0

# 3. Group frames into unique 'Events'
# This creates a unique ID for every contiguous block of 1s or 0s
df['event_id'] = (df['BlinkLabel'] != df['BlinkLabel'].shift()).cumsum()

# 4. Identify the "Peak Strength" of each event
# We only care about groups where BlinkLabel is 1
blink_groups = df[df['BlinkLabel'] == 1].groupby('event_id')

# For every frame in a blink, map the MAX velocity/acceleration reached during that blink
df['peak_vel_reached'] = blink_groups['Velocity'].transform(lambda x: x.abs().max())
df['peak_accel_reached'] = blink_groups['Acceleration'].transform(lambda x: x.abs().max())
df['frame_count'] = blink_groups['BlinkLabel'].transform('count')

# 5. Apply the "Refined Heuristic"
# RULE A: If a blink event never reached the Velocity OR Acceleration threshold, it's a squint/noise -> Set to 0
df.loc[
    (df['BlinkLabel'] == 1) & 
    ((df['peak_vel_reached'] < MIN_VELOCITY_FOR_BLINK) | 
     (df['peak_accel_reached'] < MIN_ACCEL_FOR_BLINK)), 
    'BlinkLabel'
] = 0

# RULE B: Anti-Flicker (Remove blinks that lasted only 1 frame)
df.loc[(df['BlinkLabel'] == 1) & (df['frame_count'] < 2), 'BlinkLabel'] = 0

# 6. Clean up and Save (APPEND MODE)
# Remove the helper columns
df_clean = df.drop(columns=['event_id', 'peak_vel_reached', 'peak_accel_reached', 'frame_count'])

file_path = 'refined_training_data.csv'

# Check if the file already exists
file_exists = os.path.isfile(file_path)

# Save with mode='a' (append)
# header=not file_exists means: "Only write the header if the file DOES NOT exist yet"
df_clean.to_csv(file_path, mode='a', index=False, header=not file_exists)

print("Refinement and Appending Complete!")
print(f"Current Batch - Original Blink Frames: {len(df[df['BlinkLabel'] == 1])}") # Note: using df before cleaning for comparison
print(f"Current Batch - Refined Blink Frames Added: {len(df_clean[df_clean['BlinkLabel'] == 1])}")

# To see the total size of your training set now:
total_rows = len(pd.read_csv(file_path))
print(f"Total rows now in {file_path}: {total_rows}"'''